In [3]:
using Oscar

## Code for Griffiths-Dwork Reduction

In [ ]:
# rescales a homogeneous polynomial by its pole order defined by the polynomial f and the graded ring R
function mhat(poly,f,R)
    w = sum(weights(R))[1]
    degf = total_degree(f)
    hcs = homogeneous_components(R(poly))
    return sum( hcs[k]*(k[1]+w)/degf for k in keys(hcs);init=0 )
end

# rescales a homogeneous polynomial by the reciprocal of its pole order
function mhat_Inverse(poly,f,R)
    w = sum(weights(R))[1]
    degf = total_degree(f)
    hcs = homogeneous_components(R(poly))
    return sum( hcs[k]/(k[1]+w)*degf for k in keys(hcs);init=0 )
end

# implements a single step in the reduction of pole order 
# the inputs are given as follows:
# p is a polynomial in R, a graded_polynomial_ring over K, which represents the numerator of a differential with pole locus f=0
# gb is a gr{\"o}bner basis for the jacobian ideal of factor
# C is a transformation matrix which converts quotients found with gb to the desired quotients 
function PoleReduce(p , f, R, gb, C)
    xs = gens(R)
    q,r = reduce_with_quotients(p,gb)
    q = C*transpose(q)
    q = mhat_Inverse(sum([derivative(q[i],xs[i]) for i in 1:length(xs)] ),f,R)
    return q+r
end

# calculates the total_degree of a polynomial accounting for grading
weighted_degree(p) = maximum([k[1] for k in keys(homogeneous_components(p))])

# applies PoleReduce until the polynomial is completely reduced
function PoleReduceFull(poly,f,R,gb,C)
    d = weighted_degree(R(poly))/total_degree(f)+1
    res = poly
    for _ in 1:d
        new_res = PoleReduce(res, f, R, gb, C)
        if new_res == res
            break
        end
        res = new_res
    end
    return res
end

# defines the (iterated application of derivatives to the polynomials)
# p is a polynomial in R a graded_polynomial_ring over K which represents the numerator of a differential with pole locus f=0
# us = [u1,u2,...] with ui in gens(K)
# computes the derivative \partial^n/\partial u_1 ... \partial u_n 
function ∇_GM(p,us,f,R)
    param_diff(p,u) = sum(derivative(r[1],u)*prod(gens(R).^r[2]) for r in collect(coefficients_and_exponents(p)))
    ∇(p,u) = param_diff(R(p),u) - param_diff(f,u)*mhat(p,f,R)
    pnew = p 
    for u in us
        pnew = ∇(pnew,u)
    end
    return pnew
end


# convenient functions for the analysis below
    
# the function g_y(n,m) provides numerical factor used in y_reduce below
function g_y(n,m)
    if n==0 
        out = 1
    elseif n % 2 == 0
        out= prod((n-2*i-1)//(m-i-1) for i in 0:(n//2-1))
    else
        out=0
    end
    return out
end


# computes the reduction of a polynomial y^n * p(x,z) using the reduction of pole order to eliminate all powers of y
function y_reduce(poly,f,R)
    if poly == 0 
        return 0
    else 
    w=weights(R)
    pole_ord(ints) = (sum(w[i][1]*ints[i] for i in 1:length(w)) + sum(w)[1])//total_degree(f) 
    exps = collect(exponents(poly))
    cs = collect(coefficients(poly))
    return sum(cs[i]*g_y(exps[i][2], pole_ord(exps[i]) )*x^exps[i][1]*z^exps[i][3] for i in length(cs) )
    end
end    

# computes integral of polynomial (representing the numerator of a differential) by splitting the poly into two pieces
# the first is the terms which cannot be integrated (their images under integration are not differentials on the hypersurface as poles are added)
# the second element are terms which can be integrated 
function int_dE(poly,f,R)
    dfdE = x^2*z^2 - z^4
    q,r = reduce_with_quotients(poly,[dfdE])
    return [y_reduce(r,f,R),mhat_Inverse(q[1],f,R)]
end

function int_dmu_0(poly,f,R)
    dfdmu_0 = z^4
    q,r = reduce_with_quotients(poly,[dfdmu_0])
    return [y_reduce(r,f,R),mhat_Inverse(q[1],f,R)]
end 

int_dmu_0 (generic function with 1 method)

In [ ]:
# computes derivatives of elements of R w.r.t. vars E,mu0 ,...  in K
param_diff(p,u) = sum(derivative(r[1],u)*prod(gens(R).^r[2]) for r in collect(coefficients_and_exponents(p)))

param_diff (generic function with 1 method)

## Quantum periods

In [ ]:
# Define the coefficient field and its variables
T, (mu0, mu1, mu2, E) = polynomial_ring(QQ, [:mu0, :mu1, :mu2, :E])
K = fraction_field(T)


### define polynomial ring and weighted homogeneous polynomial f
R, (x, y, z) = graded_polynomial_ring(K, [:x, :y, :z]; weights = [1,2,1])


# the old polynomial
# f = K(1//2)*y^2 +
#     x^4 -
#     mu2*x^3*z +
#     (E - K(2))*x^2*z^2 +
#     (mu1 + mu2)*x*z^3 +
#     (mu0 - E + K(1))*z^4

#  treat mu1 and mu2 as order hbar quantum corrections
f = K(1//2)*y^2 + x^4 - (E - K(2))*x^2*z^2  + (mu0 - E + K(1))*z^4

# the quantum corrections
# for convenience consider mu2 -> i mu2 and mu1 -> i mu1 
# this ensures the trick used to account for the i appearing in the recurrence for the quantum periods still applies
v1 = mu2*x^3*z + (mu1 + mu2)*x*z^3 

# compute ideal and gb plus a transformation matrix (i believe default ordering is degrevlex)
J = ideal(R, [derivative(f,xx) for xx in gens(R)])
gb,C = groebner_basis_with_transformation_matrix(J)


(Gröbner basis with 5 elements w.r.t. wdegrevlex([x, y, z], [1, 2, 1]), [0 0 1 (E-2)*z (E^2-4*E+4)*x*z; 1 0 0 0 0; 0 -1 0 2*x (2*E-4)*x^2+(4*mu0-E^2)*z^2])

In [79]:
f

x^4 + (-E + 2)*x^2*z^2 + 1//2*y^2 + (mu0 - E + 1)*z^4

In [ ]:
# compute quantum periods using above data.
# note that output is a vector of length max_iter which consists of vectors of the form [p_1,p_2] 
# where the p_1 is the integral of dPi/dE w.r.t. E and p_2 is the piece with no simple integral 

function Pi_gen(max_iter,f,R,gb,C)

    quarter = K(1//4)
    half    = K(1//2)

    delta1(U) = z * (derivative(U, x) - derivative(f, x) * mhat(U,f,R))
    delta2(U) = derivative(U, y) - derivative(f, y) * mhat(U,f,R)

    # g1 = i * g1tilde
    g_1(U) = y * ((z^2 - x^2) * delta1(U) -
                      z * x * y * delta2(U)) 

    g_2(U) = (half*z^4 - z^2*x^2 + half*x^4) * delta1(delta1(U)) +
            y*z*(-z^2*x + x^3) * delta2(delta1(U)) +
            half*y^2*x^2 * delta2(z^2 * delta2(U)) +
            half*z*(-z^2 + x^2) * (y*z*delta2(U) + x*delta1(U))
    
    v1 = (mu2*x^3*z + (mu1 + mu2)*x*z^3)
 
    b_size = 1+max_iter
    a2 = R(1)
    a1 = R(0)
    b = Vector{typeof([a1,a2])}(undef, b_size)
    b[1] = [a2,R(0)]
    # b[1] = R(1)
    # a = [a2,a1]
    prev_prev = a1
    prev = a2
    for i in 3:2:(max_iter+2)
        # i is odd: compute a[i]
        curr_odd = -g_1(prev) - v1*prev + g_2(prev_prev)
        out = int_dE(curr_odd,f,R)
        b[i-1] = [PoleReduceFull(R(out[2]),f,R,gb,C),out[1]]
        # i+1 is even: compute a[i+1] and store in b
        if i + 1 <= max_iter+2
            curr_even = g_1(curr_odd) + v1*curr_odd + g_2(prev)
            out = int_dE(curr_even,f,R)
            b[i] = [PoleReduceFull(R(out[2]),f,R,gb,C),out[1]]
            
            # Update state for next iteration
            prev_prev = curr_odd
            prev = curr_even
        end
    end
    return b 
end

Pi_gen (generic function with 1 method)

In [ ]:
# run to compile
@time Pi_gen(3,f,R,gb,C)

  0.430593 seconds (1.91 M allocations: 87.239 MiB, 92.17% compilation time)


4-element Vector{Vector{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}}:
 [1, 0]
 [0, (-mu1 - mu2)*x*z^3]
 [(-2*mu0*mu1*mu2 - 1//2*mu0*mu2^2*E - 2*mu0*mu2^2 - 1//3*mu0*E + 5//3*mu0 - 1//2*mu1^2*E + mu1^2 + 2*mu1*mu2 + 1//2*mu2^2*E^2 - 1//2*mu2^2*E + 2*mu2^2 + 1//4*E^2 - 5//3*E + 4//3)//(E - 2)*z^4 + (1//2*mu1*mu2 + 3//4*mu2^2 - 1//4)//(E - 2), (-mu1^2 - 4*mu1*mu2 - 4*mu2^2)*z^8]
 [0, (mu1^3 + 6*mu1^2*mu2 + 12*mu1*mu2^2 + 8*mu2^3)*x*z^11]

In [ ]:
# generate as many terms as needed (note seems to use a lot of RAM)
@time Pi=Pi_gen(9,f,R,gb,C)

143.337113 seconds (219.70 M allocations: 55.879 GiB, 2.35% gc time)


10-element Vector{Vector{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}}:
 [1, 0]
 [0, (-mu1 - mu2)*x*z^3]
 [(-2*mu0*mu1*mu2 - 1//2*mu0*mu2^2*E - 2*mu0*mu2^2 - 1//3*mu0*E + 5//3*mu0 - 1//2*mu1^2*E + mu1^2 + 2*mu1*mu2 + 1//2*mu2^2*E^2 - 1//2*mu2^2*E + 2*mu2^2 + 1//4*E^2 - 5//3*E + 4//3)//(E - 2)*z^4 + (1//2*mu1*mu2 + 3//4*mu2^2 - 1//4)//(E - 2), (-mu1^2 - 4*mu1*mu2 - 4*mu2^2)*z^8]
 [0, (mu1^3 + 6*mu1^2*mu2 + 12*mu1*mu2^2 + 8*mu2^3)*x*z^11]
 [(7//64*mu0^5*mu2^4 + 1//4*mu0^5*mu2^2 + 5//48*mu0^5 + 9//32*mu0^4*mu1^2*mu2^2 + 1//4*mu0^4*mu1^2 + 29//96*mu0^4*mu1*mu2^3*E + 7//48*mu0^4*mu1*mu2^3 + 25//48*mu0^4*mu1*mu2*E - 13//24*mu0^4*mu1*mu2 + 5//256*mu0^4*mu2^4*E^2 - 95//384*mu0^4*mu2^4*E + 37//96*mu0^4*mu2^4 + 7//64*mu0^4*mu2^2*E^2 - 121//96*mu0^4*mu2^2*E + 113//96*mu0^4*mu2^2 + 17//320*mu0^4*E^2 - 95//96*mu0^4*E + 253//192*mu0^4 + 7//64*mu0^3*mu1^4 + 29//96*mu0^3*mu1^3*mu2*E + 13

In [ ]:
# my laptop did not seem to garbage collect properly so I did it manually with the following command
GC.gc()

In [ ]:
# possibly useful alternate expression:
∇_GM(int_dE(param_diff(v1,mu1),f,R)[1],[mu0,mu0,mu0],f,R)

-24*x*z^15

In [107]:
PoleReduceFull(x*z^11,f,R,gb,C)

0

In [ ]:
# look at the terms which are not easily integrated w.r.t. E 
[p[2] for p in Pi]

10-element Vector{MPolyDecRingElem{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}, AbstractAlgebra.Generic.MPoly{AbstractAlgebra.Generic.FracFieldElem{QQMPolyRingElem}}}}:
 0
 (-mu1 - mu2)*x*z^3
 (-mu1^2 - 4*mu1*mu2 - 4*mu2^2)*z^8
 (mu1^3 + 6*mu1^2*mu2 + 12*mu1*mu2^2 + 8*mu2^3)*x*z^11
 -1//6*z^8
 (1//2*mu1 + mu2)*x*z^11
 -1//8*z^12
 (1//2*mu1 + mu2)*x*z^15
 -1//10*z^16
 (1//2*mu1 + mu2)*x*z^19